In [1]:
# !pip install friends

import pandas as pd
import numpy as np
from collections import defaultdict
from tqdm import tqdm 

In [ ]:
""" CODE TO CREATE FOLDERS FOR DATASETS """
# import os
# grades = ['2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12']
# datasets = ['original', 'cleaned', 'cleaned_doubled']
# # create folders for each dataset, then each grade within
# for dataset in datasets:
#     if not os.path.exists(f"comparative_datasets/{dataset}"):
#         os.makedirs(f"comparative_datasets/{dataset}")
#     for grade in grades:
#         if not os.path.exists(f"comparative_datasets/{dataset}/{grade}"):
#             os.makedirs(f"comparative_datasets/{dataset}/{grade}")

In [ ]:
""" This was just an inspection to check which grades were """
# # randomly sample rows until the difference between src and tgt FKGL grade is more than 3
# for row in full_raw.sample(frac=1).iterrows():
#     # only go into for loop if difference between src and tgt FKGL grade is more than 3
#     if abs(row[1]['abs_src_FKGL_Grade'] - row[1]['abs_tgt_FKGL_Grade']) > 1 and abs(row[1]['abs_src_FKGL_Grade'] - row[1]['abs_tgt_FKGL_Grade']) < 10:
#         for column in row[1].items():
#             print(f"{column[0]}: {column[1]}")  # print first 100 characters of each column
#         break


src: Because the material is very small, it becomes suspended in river water making the water appear cloudy, which is sometimes known as glacial milk.
tgt: Because the material is very small, it is suspended in river water making the water appear cloudy.
abs_src_FKGL_Grade: 12
abs_tgt_FKGL_Grade: 10


# Examples from above cell

src: The Belcea Quartet is a string quartet, formed in 1994, under the leadership of violinist Corina Belcea.

tgt: The Belcea Quartet is a string quartet which was formed in 1994. The leader (first violinist) of the quartet is Corina Belcea.

abs_src_FKGL_Grade: 11

abs_tgt_FKGL_Grade: 6

--------------------------------------------------------------------

src: The host city of the Games will be announced at the 121st IOC Session (which will also be the 13th Olympic Congress) to be held in Copenhagen, Denmark, on 2 October 2009.

tgt: A host city will be announced in Copenhagen, Denmark, on October 2, 2009.

abs_src_FKGL_Grade: 12

abs_tgt_FKGL_Grade: 6

--------------------------------------------------------------------

src: Over the years, the number of judges has fluctuated between three and eight, including judges in the UK and South Africa.

tgt: There have been three to eight judges, including judges in the UK and South Africa.

abs_src_FKGL_Grade: 8

abs_tgt_FKGL_Grade: 3

--------------------------------------------------------------------

src: Because the material is very small, it becomes suspended in river water making the water appear cloudy, which is sometimes known as glacial milk.

tgt: Because the material is very small, it is suspended in river water making the water appear cloudy.

abs_src_FKGL_Grade: 12

abs_tgt_FKGL_Grade: 10

In [ ]:
""" This cell combines train, test, and val splits, calculates similarity scores, grade differences, length difference ratios, and classifies the simplifications into 4 categories
This is therefore the processing pipeline from raw data to the full labelled combined dataset.
Further cleaning and filtering is done to split into different datasets for comparison
Including doubling the dataset by flipping src and tgt for all but the 'inadequate' simplifications
Final datasets will be {'original', 'cleaned', 'cleaned_doubled'}
Each dataset will have subfolders for each target grade from 2 to 12, as well as a folder for the combined baseline dev set, sampled from the individual grade sets."""

# def collate_src_tgt_info(split: str) -> pd.DataFrame:
#     assert split in ['train', 'test', 'val'], f"Invalid split: {split}. Choose from ['train', 'test', 'valid']"
    
#     path = f'data/wikilarge/raw'
#     with open(f'{path}/wiki_{split}.src', 'r', encoding='utf-8') as f:
#         src = [line.strip() for line in f.readlines()]
#     src = pd.DataFrame(src, columns=['src'])
#     with open(f'{path}/wiki_{split}.tgt', 'r', encoding='utf-8') as f:
#         tgt = [line.strip() for line in f.readlines()]
#     tgt = pd.DataFrame(tgt, columns=['tgt'])
#     infos  = pd.read_csv(f'{path}/grade_ratio_wiki_{split}.csv')
#     data = pd.concat([src, tgt, infos], axis=1)
#     print(f"loaded {data.shape} samples from {split} split.")
#     return data

# test_val_raw = pd.concat([collate_src_tgt_info(split) for split in ['test', 'val']], ignore_index=True)

# print(f"Loaded {test_val_raw.shape} samples in combined_data.")
# test_val_raw.columns

# tqdm.pandas()
# #!python -m spacy download en_core_web_sm
# import spacy
# nlp = spacy.load("en_core_web_sm")
# import en_core_web_sm
# nlp = en_core_web_sm.load()

# def calculate_similarity(data: pd.DataFrame, split: str):
#     texts = data[['src', 'tgt']].copy()
#     texts['src'] = texts['src'].progress_apply(lambda x: nlp(x))
#     texts['tgt'] = texts['tgt'].progress_apply(lambda x: nlp(x))
#     texts['similarity'] = texts.progress_apply(lambda x: x['src'].similarity(x['tgt']), axis=1)
#     data['similarity'] = texts['similarity']
#     data.to_csv(f'data/wikilarge/intermediate/wiki_{split}.csv', index=False)
#     print(f"Similarity scores calculated and saved for {split} split.")

# calculate_similarity(test_val_raw, 'test_val')

# # read in train and test_val_raw and merge them
# train = pd.read_csv('data/wikilarge/intermediate/wiki_train.csv')
# test_val = pd.read_csv('data/wikilarge/intermediate/wiki_test_val.csv')
# # merge the two dataframes
# combined_data = pd.concat([train, test_val], ignore_index=True)
# combined_data.to_csv('data/wikilarge/intermediate/wiki_combined.csv', index=False)

# # step 1 similarity done
# combined_data = pd.read_csv('data/wikilarge/intermediate/wiki_combined.csv')
# print(f"Combined data shape: {combined_data.shape}")
# combined_data.head()

# # step 2 grade_difference done
# preprocessed_data = combined_data.copy()
# grade_diff = preprocessed_data.abs_tgt_FKGL_Grade - preprocessed_data.abs_src_FKGL_Grade
# preprocessed_data['grade_diff'] = grade_diff

# # step 3 length_difference_ratio done
# src_length = preprocessed_data['src'].apply(lambda x: len(x.split()))
# tgt_length = preprocessed_data['tgt'].apply(lambda x: len(x.split()))
# length_difference_ratio = (tgt_length - src_length) / src_length
# preprocessed_data['length_difference_ratio'] = length_difference_ratio

# print(f"Data shape after adding grade_diff and length_difference_ratio: {preprocessed_data.shape}")
# print(preprocessed_data[['similarity', 'grade_diff', 'length_difference_ratio']].head(10))

# # step 4 classifier done
# from sklearn.model_selection import train_test_split
# from lazypredict.Supervised import LazyClassifier

# clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)

# sample_data = pd.read_csv('data/wikilarge/intermediate/wiki_train_labelled.csv').drop(columns=["(2, 'label')", "(3, 'label')", "(4, 'label')", "(5, 'label')",
#        "(6, 'label')", "(7, 'label')", "(8, 'label')", "(2, 'label').1",
#        "(3, 'label').1", "(4, 'label').1", "(5, 'label').1", "(6, 'label').1"])

# src_length = sample_data['src'].apply(lambda x: len(x.split()))
# tgt_length = sample_data['tgt'].apply(lambda x: len(x.split()))
# length_difference_ratio = (tgt_length - src_length) / src_length
# sample_data['length_difference_ratio'] = length_difference_ratio
# sample_data['grade_difference'] = sample_data['abs_tgt_FKGL_Grade'] - sample_data['abs_src_FKGL_Grade']

# regression_set = sample_data[['length_difference_ratio', 'similarity', 'grade_difference']].copy()
# regression_labels = sample_data['label'].copy()

# x = regression_set.to_numpy()
# y = regression_labels.to_numpy()

# X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=42)

# model_dictionary = clf.provide_models(X_train, X_test, y_train, y_test)

# bnb = model_dictionary['BernoulliNB']
# print(f"Model: {bnb.__class__.__name__}")
# print(f"Accuracy: {bnb.score(X_test, y_test)}")

# full_regression_set = preprocessed_data[['length_difference_ratio', 'similarity', 'grade_diff']].copy()
# full_regression_labels = bnb.predict(full_regression_set.to_numpy())
# preprocessed_data['label'] = full_regression_labels
# print(preprocessed_data.label.value_counts())

# postprocessed_data = preprocessed_data.copy()

# postprocessed_data.to_csv('data/wikilarge/intermediate/wiki_combined_labelled.csv', index=False)

loaded (755, 32) samples from test split.
loaded (731, 32) samples from val split.
Loaded (1486, 32) samples in combined_data.


Index(['src', 'tgt', 'current_line', 'New Line', 'Line', 'abs_src_Length',
       'abs_src_MaxDepDepth', 'abs_src_MaxDepLength', 'abs_src_DiffWords',
       'abs_src_Leven', 'abs_src_WordCount', 'abs_tgt_Length',
       'abs_tgt_MaxDepDepth', 'abs_tgt_MaxDepLength', 'abs_tgt_DiffWords',
       'abs_tgt_Leven', 'abs_tgt_WordCount', 'Length_ratio',
       'MaxDepDepth_ratio', 'MaxDepLength_ratio', 'DiffWords_ratio',
       'Leven_ratio', 'WordCount_ratio', 'abs_src_FreqRank',
       'abs_tgt_FreqRank', 'FreqRank_ratio', 'abs_src_FKGL_Grade',
       'abs_tgt_FKGL_Grade', 'FKGL_Grade_ratio', 'abs_src_ARI_Grade',
       'abs_tgt_ARI_Grade', 'ARI_Grade_ratio'],
      dtype='object')

In [ ]:
""" block for loading 3 combined datasets, before grade splitting """

""" Step 1: Read in the full labelled combined dataset, filter out rows labelled as 0 ('inadequte' simplifications) """
# original_data = pd.read_csv('data/wikilarge/intermediate/wiki_combined_labelled.csv')
# print(f"Postprocessed data shape: {original_data.shape}")

""" Cleaning data rows labelled as 0 ('inadequte' simplifications) """
# # filter out rows with label 0
# cleaned_data = original_data.copy()[original_data['label'] != 0].reset_index(drop=True)
# print(f"Postprocessed data shape: {original_data.shape}")
# print(f"Cleaned data shape: {cleaned_data.shape}")

""" For data augmentation, flip src and tgt columns to double the dataset """
# def flip_and_double_data(original_data: pd.DataFrame) -> pd.DataFrame:
#     src_columns = [column for column in original_data.columns if 'src' in column]
#     tgt_columns = [column for column in original_data.columns if 'tgt' in column]

#     # copy filtered data and flip src and tgt columns
#     flipped_data = original_data.copy()
#     for src_col, tgt_col in zip(src_columns, tgt_columns):
#         flipped_data[src_col], flipped_data[tgt_col] = flipped_data[tgt_col], flipped_data[src_col]

#     cleaned_doubled_data = pd.concat([original_data, flipped_data], ignore_index=True)

#     return cleaned_doubled_data

# cleaned_doubled_data = flip_and_double_data(cleaned_data.copy())

# print(f"Original data shape: {original_data.shape}")
# print(f"Cleaned data shape: {cleaned_data.shape}")
# print(f"Cleaned and Doubled data shape: {cleaned_doubled_data.shape}")

""" Save the 3 datasets to their respective folders """
# datasets = {'original': original_data,
#             'cleaned': cleaned_data,
#             'cleaned_doubled': cleaned_doubled_data}

# for name, dataset in datasets.items():
#     print(f"Processing dataset: {name}")
#     print(f"Dataset shape: {dataset.shape}")
#     dataset.to_csv(f'comparative_datasets/{name}/combined_dataset.csv', index=False)

Postprocessed data shape: (218367, 36)
Postprocessed data shape: (218367, 36)
Cleaned data shape: (146097, 36)
Original data shape: (218367, 36)
Cleaned data shape: (146097, 36)
Cleaned and Doubled data shape: (292194, 36)


In [ ]:
comparative_datasets = ['original', 'cleaned', 'cleaned_doubled']
for dataset in comparative_datasets:
    
    

Processing dataset: original
Dataset shape: (218367, 36)
Processing dataset: cleaned
Dataset shape: (146097, 36)
Processing dataset: cleaned_doubled
Dataset shape: (292194, 36)


In [ ]:
""" This cell is used to verify number of samples in uploaded datasets """
# from datasets import load_dataset
# lengths = []
# # Login using e.g. `huggingface-cli login` to access this dataset
# # load dataset from huggingface
# for grade in range(2, 13):
#     data = load_dataset(f"williamplacroix/wikilarge_grade{grade}_alpaca")
#     lengths.append(data['train'].num_rows)
#     print(f"Loaded data for grade {grade}: {data.shape}")
# print(lengths)